# 11 — LSTM Training & Confidence-Weighted Labels

**Question:** Does confidence-weighted training improve stage classification?

Permit-derived labels have varying quality: parcels with a full permit chain
get weight 1.0, partial permits get 0.6, and inferred-only labels get 0.3.
This notebook trains two models (weighted vs unweighted) and compares them.

**Pipeline connection:** Best model saved to `ml/models/fusion_classifier.keras` for `ml/inference/rebuild.py`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "ml" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Rebuild stage labels
STAGE_NAMES = ["destroyed", "cleared_lot", "framing", "structure_substantial", "rebuild_complete"]
N_CLASSES = len(STAGE_NAMES)

# Confidence weights for label quality tiers
CONFIDENCE_WEIGHTS = {
    "high": 1.0,     # Full permit chain
    "medium": 0.6,   # Partial permit info
    "low": 0.3,      # Inferred from SAR only
}

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## Prepare input sequences

Each parcel gets a 5-timestep sequence with 3 features per step:
- `vv_norm`: VV dB normalized to pre-fire baseline
- `vv_delta_from_prev`: change from previous timestep (captures trajectory direction)
- `doy_frac`: day-of-year / 365 (captures seasonality)

Plus 2 parcel-level context features:
- `parcel_area_m2`: larger parcels have more SAR pixels (more reliable stats)
- `sar_pixel_count`: actual number of pixels inside parcel boundary

In [ ]:
def prepare_sequences(data_dir: Path):
    """Build LSTM input arrays from zonal stats and parcel labels.
    
    Returns:
        X_seq: (n, 5, 3) — temporal features per timestep
        X_parcel: (n, 2) — parcel-level context
        y: (n,) — integer stage labels 0-4
        sample_weights: (n,) — confidence weights per sample
        parcel_ids: (n,) — for traceability
    """
    # ----- Load data -----
    zonal = pd.read_parquet(data_dir / "tabular" / "sar_zonal_stats.parquet")
    labels = pd.read_parquet(data_dir / "tabular" / "parcel_labels.parquet")

    # ----- VV mean per parcel per date -----
    vv = zonal[zonal["raster"].str.contains("vv", case=False)].copy()
    vv_pivot = vv.pivot_table(
        index="parcel_id", columns="date", values="mean", aggfunc="first"
    ).sort_index(axis=1)

    # Pixel count per parcel (from any date — should be consistent)
    pixel_counts = vv.groupby("parcel_id")["pixel_count"].first()

    # ----- Normalize VV to pre-fire baseline -----
    baseline_col = [c for c in vv_pivot.columns if "2021" in str(c)][0]
    vv_norm = vv_pivot.subtract(vv_pivot[baseline_col], axis=0)

    # ----- Compute delta from previous timestep -----
    vv_delta = vv_norm.diff(axis=1).fillna(0.0)

    # ----- Day-of-year fraction -----
    # Approximate DOY for each observation date
    date_strings = ["2021-11", "2022-01", "2022-06", "2023-06", "2024-06"]
    doy_values = [305 / 365, 15 / 365, 166 / 365, 166 / 365, 166 / 365]  # Nov, Jan, Jun, Jun, Jun

    # ----- Merge with labels -----
    # Keep only parcels that have both VV data and stage labels
    common_ids = vv_norm.index.intersection(labels.set_index("parcel_id").index)
    labels_indexed = labels.set_index("parcel_id")

    # Filter to parcels with rebuild_stage labels
    if "rebuild_stage" in labels_indexed.columns:
        labeled_ids = labels_indexed.loc[
            labels_indexed["rebuild_stage"].notna()
        ].index.intersection(common_ids)
    else:
        raise ValueError("No rebuild_stage column in parcel_labels.parquet")

    print(f"Parcels with VV data: {len(vv_norm)}")
    print(f"Parcels with stage labels: {len(labeled_ids)}")

    # ----- Build arrays -----
    n = len(labeled_ids)
    X_seq = np.zeros((n, 5, 3), dtype=np.float32)
    X_parcel = np.zeros((n, 2), dtype=np.float32)
    y = np.zeros(n, dtype=np.int32)
    sample_weights = np.zeros(n, dtype=np.float32)
    parcel_ids = []

    stage_to_int = {name: i for i, name in enumerate(STAGE_NAMES)}

    for i, pid in enumerate(labeled_ids):
        # Temporal features
        vv_norm_vals = vv_norm.loc[pid].values.astype(np.float32)
        vv_delta_vals = vv_delta.loc[pid].values.astype(np.float32)
        doy_arr = np.array(doy_values, dtype=np.float32)

        X_seq[i, :, 0] = vv_norm_vals
        X_seq[i, :, 1] = vv_delta_vals
        X_seq[i, :, 2] = doy_arr

        # Parcel context
        row = labels_indexed.loc[pid]
        X_parcel[i, 0] = row.get("parcel_area_m2", 500.0)  # default if missing
        X_parcel[i, 1] = pixel_counts.get(pid, 10)  # default if missing

        # Label
        y[i] = stage_to_int[row["rebuild_stage"]]

        # Confidence weight
        confidence = row.get("label_confidence", "low")
        sample_weights[i] = CONFIDENCE_WEIGHTS.get(confidence, 0.3)

        parcel_ids.append(pid)

    # Normalize parcel features to [0, 1]
    for j in range(X_parcel.shape[1]):
        col_min, col_max = X_parcel[:, j].min(), X_parcel[:, j].max()
        if col_max > col_min:
            X_parcel[:, j] = (X_parcel[:, j] - col_min) / (col_max - col_min)

    parcel_ids = np.array(parcel_ids)
    return X_seq, X_parcel, y, sample_weights, parcel_ids


# ----- Run -----
X_seq, X_parcel, y, sample_weights, parcel_ids = prepare_sequences(DATA_DIR)

print(f"\nX_seq shape:   {X_seq.shape}  (parcels, timesteps, features)")
print(f"X_parcel shape: {X_parcel.shape}  (parcels, context features)")
print(f"y shape:        {y.shape}  unique values: {np.unique(y)}")
print("\nStage distribution:")
for i, name in enumerate(STAGE_NAMES):
    count = (y == i).sum()
    avg_w = sample_weights[y == i].mean() if count > 0 else 0
    print(f"  {name:>25s}: {count:4d} parcels, avg confidence weight: {avg_w:.2f}")

# ----- Train/val split (spatial would be better, using stratified here) -----
(
    X_seq_train, X_seq_val,
    X_par_train, X_par_val,
    y_train, y_val,
    w_train, w_val,
) = train_test_split(
    X_seq, X_parcel, y, sample_weights,
    test_size=0.2, random_state=42, stratify=y,
)
print(f"\nTrain: {len(y_train)}, Val: {len(y_val)}")

# One-hot encode labels for categorical crossentropy
y_train_oh = keras.utils.to_categorical(y_train, N_CLASSES)
y_val_oh = keras.utils.to_categorical(y_val, N_CLASSES)

## Build two-input LSTM model

Architecture:
- **LSTM branch**: processes the 5-timestep VV sequence (captures trajectory shape)
- **Parcel branch**: encodes parcel area and pixel count (provides scale context)
- **Merge**: concatenate both branches, then Dense layers to 5-class softmax

In [ ]:
def build_lstm_model(
    seq_shape=(5, 3),
    parcel_shape=(2,),
    lstm_units=32,
    lstm_dropout=0.2,
    parcel_units=16,
    merged_units=32,
    merged_dropout=0.3,
    n_classes=5,
    learning_rate=1e-3,
):
    """Build a two-input Keras model: LSTM for time series + Dense for parcel context."""

    # ----- LSTM branch -----
    seq_input = layers.Input(shape=seq_shape, name="seq_input")
    x = layers.LSTM(lstm_units, dropout=lstm_dropout, name="lstm")(seq_input)

    # ----- Parcel context branch -----
    parcel_input = layers.Input(shape=parcel_shape, name="parcel_input")
    p = layers.Dense(parcel_units, activation="relu", name="parcel_dense")(parcel_input)

    # ----- Merge -----
    merged = layers.Concatenate(name="merge")([x, p])
    merged = layers.Dense(merged_units, activation="relu", name="merged_dense")(merged)
    merged = layers.Dropout(merged_dropout, name="merged_dropout")(merged)
    output = layers.Dense(n_classes, activation="softmax", name="stage_output")(merged)

    model = keras.Model(inputs=[seq_input, parcel_input], outputs=output)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


model_weighted = build_lstm_model()
model_weighted.summary()

## Train with sample weights (MLflow tracked)

Parcels with high-confidence labels (full permit chain) contribute more to the
loss than parcels with inferred labels. This prevents the model from overfitting
to noisy labels.

In [ ]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("marshall-fire-rebuild")

EPOCHS = 30
BATCH_SIZE = 64

callbacks_weighted = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
]

with mlflow.start_run(run_name="lstm_weighted"):
    mlflow.log_params({
        "model": "lstm_two_input",
        "lstm_units": 32,
        "lstm_dropout": 0.2,
        "parcel_units": 16,
        "merged_units": 32,
        "merged_dropout": 0.3,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": 1e-3,
        "sample_weighted": True,
        "confidence_weights": str(CONFIDENCE_WEIGHTS),
        "n_train": len(y_train),
        "n_val": len(y_val),
    })

    history_weighted = model_weighted.fit(
        [X_seq_train, X_par_train],
        y_train_oh,
        sample_weight=w_train,
        validation_data=([X_seq_val, X_par_val], y_val_oh),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks_weighted,
        verbose=1,
    )

    # Log epoch metrics
    for i, (loss, acc, vloss, vacc) in enumerate(zip(
        history_weighted.history["loss"],
        history_weighted.history["accuracy"],
        history_weighted.history["val_loss"],
        history_weighted.history["val_accuracy"],
    )):
        mlflow.log_metrics(
            {"train_loss": loss, "train_acc": acc, "val_loss": vloss, "val_acc": vacc},
            step=i + 1,
        )

    best_val_acc_weighted = max(history_weighted.history["val_accuracy"])
    mlflow.log_metric("best_val_accuracy", best_val_acc_weighted)
    print(f"\nWeighted model — best val accuracy: {best_val_acc_weighted:.4f}")

## Evaluate -- classification report and confusion matrix

The confusion matrix reveals which stage pairs the model confuses.
Expected: cleared_lot vs framing will have the most confusion because
their SAR signatures overlap (both are low-backscatter surfaces).

In [ ]:
# ----- Predict on validation set -----
y_pred_proba = model_weighted.predict([X_seq_val, X_par_val])
y_pred = np.argmax(y_pred_proba, axis=1)

# ----- Classification report -----
print("=" * 60)
print("WEIGHTED MODEL — Classification Report")
print("=" * 60)
print(
    classification_report(
        y_val, y_pred, target_names=STAGE_NAMES, zero_division=0
    )
)

# ----- Confusion matrix heatmap -----
cm = confusion_matrix(y_val, y_pred)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=STAGE_NAMES,
    yticklabels=STAGE_NAMES,
    ax=ax,
    linewidths=0.5,
    linecolor="white",
)
ax.set_xlabel("Predicted Stage", fontsize=12)
ax.set_ylabel("True Stage", fontsize=12)
ax.set_title("Weighted LSTM — Confusion Matrix", fontsize=14, fontweight="bold")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()

# Log to MLflow (still in the weighted run context)
with mlflow.start_run(run_name="lstm_weighted", nested=True):
    val_f1 = f1_score(y_val, y_pred, average="weighted", zero_division=0)
    mlflow.log_metric("val_f1_weighted", val_f1)
    mlflow.log_figure(fig, "confusion_matrix_weighted.png")
    print(f"Val F1 (weighted avg): {val_f1:.4f}")

plt.show()

## Compare weighted vs unweighted training

Train a second model without sample weights. If confidence weighting helps,
the weighted model should have higher F1 on high-confidence validation parcels
even if overall accuracy is similar.

In [ ]:
# ----- Train unweighted model -----
model_unweighted = build_lstm_model()

callbacks_unweighted = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
]

with mlflow.start_run(run_name="lstm_unweighted"):
    mlflow.log_params({
        "model": "lstm_two_input",
        "lstm_units": 32,
        "sample_weighted": False,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
    })

    history_unweighted = model_unweighted.fit(
        [X_seq_train, X_par_train],
        y_train_oh,
        # NO sample_weight here
        validation_data=([X_seq_val, X_par_val], y_val_oh),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks_unweighted,
        verbose=1,
    )

    best_val_acc_unweighted = max(history_unweighted.history["val_accuracy"])
    mlflow.log_metric("best_val_accuracy", best_val_acc_unweighted)

# ----- Compare results -----
y_pred_uw = np.argmax(
    model_unweighted.predict([X_seq_val, X_par_val]), axis=1
)
f1_uw = f1_score(y_val, y_pred_uw, average="weighted", zero_division=0)

print("\n" + "=" * 60)
print("COMPARISON: Weighted vs Unweighted")
print("=" * 60)
print(f"{'Metric':<25s} {'Weighted':>12s} {'Unweighted':>12s}")
print("-" * 50)
print(f"{'Best val accuracy':<25s} {best_val_acc_weighted:12.4f} {best_val_acc_unweighted:12.4f}")
print(f"{'Val F1 (weighted avg)':<25s} {val_f1:12.4f} {f1_uw:12.4f}")

diff = val_f1 - f1_uw
winner = "Weighted" if diff > 0 else "Unweighted" if diff < 0 else "Tie"
print(f"\nWinner: {winner} (F1 diff: {diff:+.4f})")

# ----- Per-stage comparison -----
print("\nPer-stage F1 comparison:")
f1_w_per = f1_score(y_val, y_pred, average=None, zero_division=0)
f1_uw_per = f1_score(y_val, y_pred_uw, average=None, zero_division=0)

for i, name in enumerate(STAGE_NAMES):
    print(f"  {name:>25s}: weighted={f1_w_per[i]:.3f}  unweighted={f1_uw_per[i]:.3f}  diff={f1_w_per[i]-f1_uw_per[i]:+.3f}")

## Save best model

Save the winning model (weighted or unweighted) as the production artifact.
The inference pipeline (`ml/inference/rebuild.py`) loads this file directly.

In [ ]:
# Pick the better model
if val_f1 >= f1_uw:
    best_model = model_weighted
    best_label = "weighted"
else:
    best_model = model_unweighted
    best_label = "unweighted"

save_path = MODEL_DIR / "fusion_classifier.keras"
best_model.save(save_path)
print(f"Best model ({best_label}) saved to {save_path}")
print(f"File size: {save_path.stat().st_size / 1024:.1f} KB")

# Log to MLflow
with mlflow.start_run(run_name=f"lstm_best_{best_label}"):
    mlflow.log_params({
        "model": "lstm_two_input",
        "best_variant": best_label,
        "n_classes": N_CLASSES,
        "stage_names": str(STAGE_NAMES),
    })
    mlflow.log_metric("best_val_f1", max(val_f1, f1_uw))
    mlflow.log_artifact(str(save_path), artifact_path="model")
    print("Model logged to MLflow.")
    print("View at: http://localhost:5000")